In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os
os.chdir('drive/MyDrive/AI+X고급/식물을 부탁해/kjw')

In [3]:
import json
import cv2
import random
from google.colab.patches import cv2_imshow
import matplotlib.pyplot as plt

## 데이터 랜덤 추출(일단 연습이니까 train데이터셋만 사용. 그걸 6(훈련):2(검증):2(테스트)로 나눔)

### 1-1.식물 이미지에서 YOLO로 잎 이미지 추출을 위한 데이터셋 만들기
### 1-2. 추출한 잎 이미지로 건조/일반/과습 분류을 위한 데이터셋 만들기

##바로 아래 3개의 셀에서 식물종 변경 필수!!!!!

# 식물종 명시(필수!!!!!)

In [4]:
plant_name = "관음죽"
label_path = "TL_B.관상수_2.중생식물_13.관음죽"

# 원본 이미지/라벨 경로
img_base  = f"/content/관음죽_dataset/train"      # normal/overwatered/underwatered 폴더 포함
json_base = f"/content/TL_B.관상수_2.중생식물_13.관음죽"     # 라벨(JSON) 폴더 루트

In [5]:
# 이미지 데이터셋 불러오기
# (1) 드라이브 -> Colab으로 복사
!cp "/content/drive/MyDrive/AI+X고급/식물을 부탁해/kjw/dataset(base)/관음죽_dataset.zip" "/content/"

# (2) Colab에서 압축 해제
!unzip -q "/content/관음죽_dataset.zip" -d "/content/관음죽_dataset"

# (3) 압축 파일 삭제 (용량 절약)
!rm "/content/관음죽_dataset.zip"

In [6]:
# 라벨링 데이터셋 불러오기
# (1) 드라이브 -> Colab으로 복사
!cp "/content/drive/MyDrive/AI+X고급/식물을 부탁해/labelling_data/TL_B.관상수_2.중생식물_13.관음죽.zip" "/content/"

# (2) Colab에서 압축 해제
!unzip -q "/content/TL_B.관상수_2.중생식물_13.관음죽.zip" -d "/content/TL_B.관상수_2.중생식물_13.관음죽"

# (3) 압축 파일 삭제 (용량 절약)
!rm "/content/TL_B.관상수_2.중생식물_13.관음죽.zip"

스트리밍 출력 내용이 길어서 마지막 5000줄이 삭제되었습니다.


### 2-2. 데이터셋 만들기

In [12]:
import os, json, cv2, random, shutil
import numpy as np

# 클래스 목록
classes = ["normal", "overwatered", "underwatered"]

# 출력 디렉토리 (이번엔 /content 밑에 바로 생성)
det_out = f"/content/{plant_name}_dataset_detection"
cls_out = f"/content/{plant_name}_dataset_classification"

# 샘플링 개수(클래스별)
n_samples = 3000

# 디렉토리 생성
for split in ["train", "val", "test"]:
    os.makedirs(os.path.join(det_out, split, "images"), exist_ok=True)
    os.makedirs(os.path.join(det_out, split, "labels"), exist_ok=True)
    for cls in classes:
        os.makedirs(os.path.join(cls_out, split, cls), exist_ok=True)

# 클래스별 처리
for cls in classes:
    cls_dir = os.path.join(img_base, cls)
    files = [f for f in os.listdir(cls_dir) if f.lower().endswith(".jpg")]

    take = min(n_samples, len(files))
    if take == 0:
        print(f"[경고] {cls} 클래스에 JPG가 없습니다. 건너뜀.")
        continue

    selected = random.sample(files, take)
    random.shuffle(selected)

    n_train = int(take * 0.6)
    n_val   = int(take * 0.2)

    splits = {
        "train": selected[:n_train],
        "val":   selected[n_train:n_train+n_val],
        "test":  selected[n_train+n_val:]
    }

    for split, split_files in splits.items():
        for fname in split_files:
            stem = os.path.splitext(fname)[0]
            img_path  = os.path.join(cls_dir, fname)
            json_path = os.path.join(json_base, stem + ".json")

            if not os.path.exists(json_path):
                continue

            img = cv2.imread(img_path)
            if img is None:
                continue
            H, W = img.shape[:2]

            with open(json_path, "r", encoding="utf-8") as f:
                data = json.load(f)

            # 1) 탐지용
            yolo_lines = []
            for ann in data.get("annotations", []):
                poly = ann.get("plant_polygon", [])
                if not poly or len(poly) < 6:
                    continue
                xs = poly[0::2]
                ys = poly[1::2]
                xmin, xmax = min(xs), max(xs)
                ymin, ymax = min(ys), max(ys)

                x_center = (xmin + xmax) / 2 / W
                y_center = (ymin + ymax) / 2 / H
                width    = (xmax - xmin) / W
                height   = (ymax - ymin) / H

                yolo_lines.append(f"0 {x_center:.6f} {y_center:.6f} {width:.6f} {height:.6f}")

            det_img_out = os.path.join(det_out, split, "images", fname)
            det_lbl_out = os.path.join(det_out, split, "labels", stem + ".txt")
            cv2.imwrite(det_img_out, img)
            with open(det_lbl_out, "w", encoding="utf-8") as out:
                out.write("\n".join(yolo_lines))

            # 2) 분류용
            for i, ann in enumerate(data.get("annotations", [])):
                poly = ann.get("plant_polygon", [])
                if not poly or len(poly) < 6:
                    continue
                xs = poly[0::2]
                ys = poly[1::2]
                xmin, xmax = int(min(xs)), int(max(xs))
                ymin, ymax = int(min(ys)), int(max(ys))

                xmin = max(0, xmin); ymin = max(0, ymin)
                xmax = min(W, xmax); ymax = min(H, ymax)

                leaf = img[ymin:ymax, xmin:xmax]
                if leaf.size == 0:
                    continue

                crop_name = f"{stem}_{i}.jpg"
                crop_out = os.path.join(cls_out, split, cls, crop_name)
                cv2.imwrite(crop_out, leaf)

# data.yaml 생성
data_yaml = f"""
path: /content/{plant_name}_dataset_detection

train: train/images
val: val/images
test: test/images

names:
  0: leaf
"""

yaml_path = os.path.join(det_out, "data.yaml")
with open(yaml_path, "w", encoding="utf-8") as f:
    f.write(data_yaml)


In [13]:
# ========= 최종 ZIP 묶기 =========
det_zip_base = f"/content/{plant_name}_dataset_detection"
cls_zip_base = f"/content/{plant_name}_dataset_classification"

for base in [det_zip_base, cls_zip_base]:
    if os.path.exists(base + ".zip"):
        os.remove(base + ".zip")

print("[ZIP] 탐지용 폴더 압축 중...")
shutil.make_archive(det_zip_base, 'zip', det_out)

print("[ZIP] 분류용 폴더 압축 중...")
shutil.make_archive(cls_zip_base, 'zip', cls_out)

print(f"✅ 완료: {det_zip_base}.zip")
print(f"✅ 완료: {cls_zip_base}.zip")

[ZIP] 탐지용 폴더 압축 중...
[ZIP] 분류용 폴더 압축 중...
✅ 완료: /content/관음죽_dataset_detection.zip
✅ 완료: /content/관음죽_dataset_classification.zip


In [14]:
zip_path = f"/content/{plant_name}_dataset_detection"

dest_path = f"/content/drive/MyDrive/AI+X고급/식물을 부탁해/kjw/dataset(detection)/"
!cp "{zip_path}.zip" "{dest_path}"

print("✅ 드라이브 복사 완료:", dest_path)

✅ 드라이브 복사 완료: /content/drive/MyDrive/AI+X고급/식물을 부탁해/kjw/dataset(detection)/


In [15]:
zip_path = f"/content/{plant_name}_dataset_classification"

dest_path = f"/content/drive/MyDrive/AI+X고급/식물을 부탁해/kjw/dataset(classification)/"
!cp "{zip_path}.zip" "{dest_path}"

print("✅ 드라이브 복사 완료:", dest_path)

✅ 드라이브 복사 완료: /content/drive/MyDrive/AI+X고급/식물을 부탁해/kjw/dataset(classification)/
